# 📊 Customer Churn Prediction – Exploratory Data Analysis

> **Project**: Telecom Customer Churn Prediction  
> **Author**: Churn ML Team  
> **Dataset**: 10,000 synthetic telecom customer records  
> **Goal**: Understand churn patterns and engineer predictive features

---

## Table of Contents
1. [Setup & Data Loading](#1-setup)
2. [Dataset Overview](#2-overview)
3. [Target Distribution](#3-target)
4. [Numerical Feature Analysis](#4-numerical)
5. [Categorical Feature Analysis](#5-categorical)
6. [Correlation Analysis](#6-correlation)
7. [Feature Engineering Preview](#7-engineering)
8. [Key Insights Summary](#8-summary)

---
## 1. Setup & Data Loading <a id='1-setup'></a>

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.data_generation import generate_dataset
from src.feature_engineering import (
    add_domain_features, split_X_y,
    CATEGORICAL_FEATURES, NUMERICAL_FEATURES
)
from src.visualize import (
    plot_churn_distribution,
    plot_numerical_distributions,
    plot_categorical_churn_rates,
    plot_correlation_heatmap,
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.3f}'.format)

print('✅ Setup complete!')

In [ ]:
# Generate (or load) the dataset
df = generate_dataset(n_samples=10_000, random_state=42,
                      output_path='../data/raw/telecom_churn.csv')
print(f'Dataset shape: {df.shape}')
print(f'Churn rate:    {df["churn"].mean():.2%}')

---
## 2. Dataset Overview <a id='2-overview'></a>

In [ ]:
df.head()

In [ ]:
print('=== DATA TYPES ===')
print(df.dtypes.to_string())
print('\n=== MISSING VALUES ===')
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
print('=== DESCRIPTIVE STATISTICS ===')
display(df.describe(include='all').T)

---
## 3. Target Distribution <a id='3-target'></a>

In [ ]:
fig = plot_churn_distribution(df, save_path='../reports/figures/eda/01_churn_distribution.png')
plt.show()

churn_counts = df['churn'].value_counts()
print(f"No Churn: {churn_counts[0]:,} ({churn_counts[0]/len(df):.1%})")
print(f"Churn:    {churn_counts[1]:,} ({churn_counts[1]/len(df):.1%})")

---
## 4. Numerical Feature Analysis <a id='4-numerical'></a>

In [ ]:
num_feats = ['tenure', 'monthly_charges', 'total_charges',
             'num_products', 'support_calls', 'avg_monthly_gb_download']

fig = plot_numerical_distributions(
    df, features=num_feats,
    save_path='../reports/figures/eda/02_numerical_distributions.png'
)
plt.show()

In [ ]:
# Statistical comparison of churned vs retained customers
print('=== Mean values split by churn label ===')
display(
    df.groupby('churn')[num_feats].mean().round(2)
      .rename(index={0: 'No Churn', 1: 'Churn'})
)

In [ ]:
# Tenure distribution insight
from scipy import stats

churn_tenure    = df[df['churn'] == 1]['tenure']
no_churn_tenure = df[df['churn'] == 0]['tenure']
t_stat, p_val   = stats.ttest_ind(churn_tenure, no_churn_tenure)

print(f'Churned customers – mean tenure:     {churn_tenure.mean():.1f} months')
print(f'Retained customers – mean tenure:    {no_churn_tenure.mean():.1f} months')
print(f't-test: t={t_stat:.2f}, p={p_val:.2e}  → {"Significant" if p_val < 0.05 else "Not significant"}')

---
## 5. Categorical Feature Analysis <a id='5-categorical'></a>

In [ ]:
cat_feats_subset = [
    'contract', 'internet_service', 'payment_method',
    'paperless_billing', 'tech_support', 'online_security',
]

fig = plot_categorical_churn_rates(
    df, features=cat_feats_subset,
    save_path='../reports/figures/eda/03_categorical_churn_rates.png'
)
plt.show()

In [ ]:
# Key categorical churn rates
print('=== Churn rate by contract type ===')
display(df.groupby('contract')['churn'].agg(['mean', 'count']).rename(columns={'mean': 'churn_rate'}))

print('\n=== Churn rate by internet service ===')
display(df.groupby('internet_service')['churn'].agg(['mean', 'count']).rename(columns={'mean': 'churn_rate'}))

---
## 6. Correlation Analysis <a id='6-correlation'></a>

In [ ]:
fig = plot_correlation_heatmap(
    df, save_path='../reports/figures/eda/04_correlation_heatmap.png'
)
plt.show()

In [ ]:
# Point-biserial correlation with churn target
num_df = df.select_dtypes(include=np.number)
corr_with_churn = num_df.corr()['churn'].drop('churn').sort_values(key=abs, ascending=False)

print('=== Correlation with Churn (sorted by |r|) ===')
display(corr_with_churn.to_frame('correlation').style.background_gradient(
    cmap='RdYlGn_r', vmin=-1, vmax=1
))

---
## 7. Feature Engineering Preview <a id='7-engineering'></a>

In [ ]:
X, y = split_X_y(df)
X_eng = add_domain_features(X)

new_features = ['lifetime_value_proxy', 'charges_per_product',
                'is_fiber_no_security', 'contract_risk', 'support_call_rate']

print('=== Engineered Features ===')
display(X_eng[new_features].describe().round(3))

In [ ]:
# Correlation of engineered features with target
eng_with_target = X_eng[new_features].copy()
eng_with_target['churn'] = y.values
eng_corr = eng_with_target.corr()['churn'].drop('churn').sort_values(key=abs, ascending=False)

print('=== Engineered Features – Correlation with Churn ===')
print(eng_corr.to_string())

---
## 8. Key Insights Summary <a id='8-summary'></a>

| # | Insight | Implication |
|---|---------|-------------|
| 1 | **Month-to-month** contract customers churn at ~3× the rate of two-year customers | Contract type is the strongest single predictor |
| 2 | **Short tenure** (< 12 months) correlates strongly with churn | Onboarding experience is critical |
| 3 | **Fiber optic** users without security add-ons have elevated churn | Upsell security products to fibre users |
| 4 | **Electronic check** payment correlates with higher churn | Auto-pay adoption campaign may help |
| 5 | **High monthly charges** increase churn probability | Value perception / pricing review needed |
| 6 | **Tech support** and **online security** add-ons reduce churn | Promote these add-ons proactively |
| 7 | **Lifetime value proxy** (tenure × charges) is highly predictive | High-LTV customers at risk need targeted retention |

---
_Proceed to `notebooks/02_model_training.ipynb` to train and evaluate models._